# LifeMate — Public test submission

Import into a **private** Kaggle notebook. Attach `amnpham/zalochallenge`, enable a GPU and Internet for setup, then run all cells in order. No Kaggle API token is required.

This notebook runs the current GitHub pipeline on **all public_test samples using audio**, with no train transcripts, labels, reference replay, or local score calculation. It does not submit automatically. Public-test labelling/tuning with external models or services is not part of this workflow.

After successful inference and metadata-only validation, download `/kaggle/working/result/submission.jsonl` and submit it yourself. Full input coverage is required before final export. Runtime/semantic failures can still score zero even when a file passes structural validation.


Pipeline defaults to **clean** with 27 seconds. Planning is optional for separate development comparisons; no automatic leaderboard submission.


In [ ]:
REPO_URL = "https://github.com/AmnO-O/ZolaAIC.git"
REVISION = "main"  # Set an exact commit hash to reproduce a source snapshot.
REPO_DIR = "/kaggle/working/ZolaAIC-public"
DATASET_ROOT = "/kaggle/input/datasets/amnpham/zalochallenge/train+public_test"

LIMIT = None  # None = FULL public test. A small number is debug-only and fails full-file export.
BUDGET_SECONDS = 27  # Internal budget for the competition target
PROMPT_MODE = "system"  # General instructions, no few-shot
PROMPT_VARIANT = "clean"  # "clean" for direct calls; "planning" for action-plan revisions
MAX_TURNS = 8  # Bounds decisions, including protocol corrections
PROMPT_CACHE = True
DETERMINISTIC = True
RUN_UNIT_TESTS = True
STEP_LOGGING = False  # Full prompt logs add overhead and can become very large.
LIVE_STEPS = False    # Requires STEP_LOGGING; mirrors detailed logs into notebook output.
PROGRESS_SECONDS = 10

# Optional existing LOCAL asset directories; blank downloads the pinned assets.
LLM_PATH = ""
ASR_PATH = ""
SPLIT = "public_test"  # Fixed inference split; there is no train-reference mode here.


## 1. Clone and install

The resolved Git commit is printed and saved with the result. A clean checkout is updated to the requested revision without merging. Local edits stop setup; preserve them and choose another `REPO_DIR`. Internet is used for source, packages, and model setup; inference sets local-only Hugging Face flags. This is not a network-disabled deployment test.


In [ ]:
import json
import os
from pathlib import Path
import subprocess
import sys

repo = Path(REPO_DIR).resolve()
if repo.exists():
    actual_root = Path(subprocess.check_output(
        ["git", "rev-parse", "--show-toplevel"], cwd=repo, text=True).strip()).resolve()
    if actual_root != repo:
        raise RuntimeError("REPO_DIR must point to the repository root")
    remote = subprocess.check_output(
        ["git", "remote", "get-url", "origin"], cwd=repo, text=True).strip()
    if remote != REPO_URL:
        raise RuntimeError(f"Unexpected repository at {repo}; choose a new REPO_DIR")
    dirty = subprocess.check_output(
        ["git", "status", "--porcelain"], cwd=repo, text=True).strip()
    if dirty:
        raise RuntimeError("Checkout has local edits. Preserve them and choose a new REPO_DIR.")
else:
    subprocess.run(["git", "clone", REPO_URL, str(repo)], check=True)

# Works for a remote branch, tag, or exact commit; never merges local changes.
subprocess.run(["git", "fetch", "origin", REVISION], cwd=repo, check=True)
subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=repo, check=True)
if not (repo / "src/lifemate/prompt_assets/system.txt").is_file():
    raise RuntimeError("This revision predates the clean pipeline; select its notebook or a newer revision.")

COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip()
print("Source commit:", COMMIT)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(repo / "requirements.txt")], check=True)


## 2. Check public input and GPU

Only `public_test.jsonl` and the public tool catalog are checked. No environment folder is enumerated or read by notebook code. App state is loaded only by the organiser ToolEnv during inference.


In [ ]:
import math
import torch

if SPLIT != "public_test":
    raise ValueError("This notebook is public_test-only")
if LIMIT is not None and (type(LIMIT) is not int or LIMIT < 1):
    raise ValueError("LIMIT must be a positive integer or None")
if not math.isfinite(BUDGET_SECONDS) or BUDGET_SECONDS <= 0:
    raise ValueError("Invalid sample budget")
if PROMPT_MODE not in {"baseline", "system"}:
    raise ValueError("Unknown prompt mode")
if LIVE_STEPS and not STEP_LOGGING:
    raise ValueError("LIVE_STEPS requires STEP_LOGGING")
if not math.isfinite(PROGRESS_SECONDS) or PROGRESS_SECONDS < 1:
    raise ValueError("PROGRESS_SECONDS must be at least 1")
if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU in Kaggle settings")
print("GPU:", torch.cuda.get_device_name(0), "Torch:", torch.__version__)

candidates = [Path(DATASET_ROOT) / SPLIT]
if DATASET_ROOT == "/kaggle/input/datasets/amnpham/zalochallenge/train+public_test":
    candidates.append(Path("/kaggle/input/zalochallenge/train+public_test/public_test"))
split_dir = next((p for p in candidates if (p / "public_test_tools_catalog.json").is_file()
                  and (p / "public_test.jsonl").is_file()), None)
if split_dir is None:
    raise FileNotFoundError("Correct DATASET_ROOT using the Kaggle Input panel")
with (split_dir / "public_test.jsonl").open(encoding="utf-8") as source:
    expected_ids = [json.loads(line)["id"] for line in source if line.strip()]
if not expected_ids or len(set(expected_ids)) != len(expected_ids):
    raise ValueError("Invalid public input ID manifest")
expected_count = len(expected_ids)
print("Public split:", split_dir)
print("Public samples:", expected_count)
print("Selected run size:", expected_count if LIMIT is None else min(LIMIT, expected_count))
print("Per-sample budget:", BUDGET_SECONDS, "seconds; competition target is under 30 seconds on A30")

if PROMPT_VARIANT == "planning" and PROMPT_MODE != "system":
    raise ValueError("Planning requires PROMPT_MODE=system")
if not isinstance(MAX_TURNS, int) or MAX_TURNS < 1:
    raise ValueError("MAX_TURNS must be positive")


## 3. Prepare pinned models

The LLM weights require roughly 8 GB, plus package/cache/workspace space. Downloads happen here. Gipformer uses CPU INT8 and Qwen uses GPU FP16. Tests use synthetic fixtures; they do not open competition environments or train labels.


In [ ]:
for component, override in (("llm", LLM_PATH), ("asr", ASR_PATH)):
    if override:
        model_path = Path(override).resolve()
    else:
        subprocess.run([sys.executable, "scripts/download_models.py", "--component", component], cwd=repo, check=True)
        model_path = repo / "models" / component
    if not model_path.is_dir():
        raise FileNotFoundError(f"Missing {component} model directory: {model_path}")
    os.environ[f"LIFEMATE_{component.upper()}_PATH"] = str(model_path)

os.environ.update(
    HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", PYTHONHASHSEED="42",
    CUBLAS_WORKSPACE_CONFIG=":4096:8", LIFEMATE_ASR_KIND="gipformer",
    LIFEMATE_ASR_PRECISION="int8", LIFEMATE_ASR_THREADS="4",
    LIFEMATE_PROMPT_MODE=PROMPT_MODE,
    LIFEMATE_PROMPT_VARIANT=PROMPT_VARIANT,
    LIFEMATE_PROMPT_CACHE="1" if PROMPT_CACHE else "0",
    LIFEMATE_DETERMINISTIC="1" if DETERMINISTIC else "0",
    LIFEMATE_STEERABILITY_PATH=str(repo / "vendor" / "steerability"),
)
if RUN_UNIT_TESTS:
    test_env = dict(os.environ, LIFEMATE_PROMPT_VARIANT="clean")
    subprocess.run([sys.executable, "-B", "-m", "unittest", "discover", "-s", "tests", "-v"], cwd=repo, env=test_env, check=True)
print("Local assets ready; inference download paths are disabled.")

# Read installed package versions in a fresh process after pip installation.
runtime_versions = json.loads(subprocess.check_output([sys.executable, "-c",
    "import importlib.metadata as m,json; print(json.dumps({p:m.version(p) for p in [\"torch\",\"transformers\",\"numpy\",\"sherpa-onnx\"]}))"], text=True))
print("Inference package versions:", runtime_versions)


## 4. Run the full public test

Progress reports completed samples and termination reasons without printing every prompt. Each sample uses one ToolEnv, and actual calls/hash are exported by the runner. There are no automatic sample retries. The inference subprocess loads models once.

A run can take hours on T4. The source uses cooperative deadlines; this notebook does not establish A30 compliance. If a session cannot export a genuine row, the process finishes other samples and exits nonzero; partial output is retained for diagnosis and cannot pass the final coverage check.


In [ ]:
if BUDGET_SECONDS >= 30:
    print("DEVELOPMENT BUDGET: results do not establish the 30-second competition requirement.")
from datetime import datetime, timezone
from collections import Counter
import time
import uuid

run_dir = Path("/kaggle/working/public_runs") / (datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8])
run_dir.mkdir(parents=True)
submission = run_dir / "submission.jsonl"
trace_path = submission.with_suffix(".trace.jsonl")
log_path = run_dir / "inference.log"
command = [sys.executable, "-B", "scripts/infer.py", "--split-dir", str(split_dir),
           "--backend", "lifemate.backend:create_backend", "--budget-seconds", str(BUDGET_SECONDS),
           "--output", str(submission)]
command += ["--prompt-variant", PROMPT_VARIANT, "--max-turns", str(MAX_TURNS)]
if LIMIT is not None:
    command += ["--limit", str(LIMIT)]
if STEP_LOGGING:
    command += ["--step-log", str(run_dir / "steps.jsonl")]
    if LIVE_STEPS:
        command.append("--verbose-steps")

completed = 0
trace_offset = log_offset = 0
terminations = Counter()

def progress():
    global completed, trace_offset, log_offset
    latest = None
    if trace_path.is_file():
        with trace_path.open(encoding="utf-8") as source:
            source.seek(trace_offset)
            while True:
                line = source.readline()
                if not line or not line.endswith("\n"):
                    break
                event = json.loads(line)
                trace_offset = source.tell()
                completed += 1
                terminations[event.get("termination", "unknown")] += 1
                latest = event
    if LIVE_STEPS and log_path.is_file():
        with log_path.open(encoding="utf-8") as source:
            source.seek(log_offset)
            text = source.read()
            log_offset = source.tell()
        if text:
            print(text, end="", flush=True)
    if latest is not None:
        print(f"Processed {completed}/{expected_count}; last={latest['id']} "
              f"status={latest.get('termination')} elapsed={latest.get('elapsed_seconds', 0):.2f}s", flush=True)
    return latest

started = time.monotonic()
last_heartbeat = started
with log_path.open("w", encoding="utf-8") as log:
    process = subprocess.Popen(command, cwd=repo, stdout=log, stderr=subprocess.STDOUT,
        env={**os.environ, "PYTHONUNBUFFERED":"1", "PYTHONIOENCODING":"utf-8"})
    while process.poll() is None:
        latest = progress()
        if latest is None and time.monotonic() - last_heartbeat >= 60:
            print(f"Still running; completed {completed}/{expected_count}. Logs: {log_path}", flush=True)
            last_heartbeat = time.monotonic()
        time.sleep(PROGRESS_SECONDS)
    returncode = process.wait()
progress()
run_report = {
    "commit": COMMIT, "split": SPLIT, "split_dir": str(split_dir), "expected_samples":expected_count,
    "limit": LIMIT, "prompt_variant": PROMPT_VARIANT, "max_turns": MAX_TURNS, "budget_seconds":BUDGET_SECONDS, "prompt_mode":PROMPT_MODE,
    "prompt_cache":PROMPT_CACHE, "deterministic":DETERMINISTIC,
    "step_logging":STEP_LOGGING, "inference_exit_code":returncode,
    "process_seconds_including_model_load_and_polling":time.monotonic()-started,
    "gpu":torch.cuda.get_device_name(0), "notebook_torch":torch.__version__,
    "installed_package_versions":runtime_versions,
    "model_config":json.loads((repo / "configs/baseline.json").read_text()),
    "termination_counts":dict(terminations),
}
(run_dir / "run_report.json").write_text(json.dumps(run_report, indent=2), encoding="utf-8")
print("Artifacts:", run_dir)
print("Termination counts:", dict(terminations))
if returncode:
    print(log_path.read_text(encoding="utf-8", errors="replace")[-6000:])
    raise RuntimeError("Inference did not finish cleanly. Partial outputs are retained; do not submit them.")


## 5. Validate and export submission.jsonl

Validation checks JSON structure, SHA-256 hash format, exactly one row for each public input ID, and tool permissions from input metadata. It never creates ToolEnv, replays calls, inspects app state, labels public data, or calculates a score. A matching hash format does not prove the predicted state is correct.

The final file is copied only after complete validation. A debug subset cannot pass. Existing final output is never replaced by a different file.


In [ ]:
import hashlib
import shutil
import statistics

validation_path = run_dir / "submission_validation.json"
validated = subprocess.run([sys.executable, "-B", "scripts/validate_submission.py",
    "--split-dir", str(split_dir), "--split", "public_test", "--predictions", str(submission),
    "--output", str(validation_path)], cwd=repo, check=False)
validation = json.loads(validation_path.read_text(encoding="utf-8"))
if validated.returncode or not validation["valid"]:
    raise RuntimeError("Submission failed validation. Inspect submission_validation.json; final file was not exported.")

traces = [json.loads(line) for line in trace_path.read_text(encoding="utf-8").splitlines() if line.strip()]
times = sorted(row["elapsed_seconds"] for row in traces if "elapsed_seconds" in row)
summary = {
    "samples":len(traces), "termination_counts":dict(Counter(row.get("termination") for row in traces)),
    "errors":sum(row.get("termination")=="error" for row in traces),
    "samples_at_or_above_30_seconds":sum(seconds >= 30 for seconds in times),
}
if times:
    summary.update(mean_seconds=sum(times)/len(times), p50_seconds=statistics.median(times),
                   p95_seconds=times[math.ceil(0.95*len(times))-1], max_seconds=times[-1])
(run_dir / "runtime_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

final_submission = Path("/kaggle/working/result/submission.jsonl")
final_submission.parent.mkdir(parents=True, exist_ok=True)
if final_submission.exists():
    if hashlib.sha256(final_submission.read_bytes()).hexdigest() != validation["sha256"]:
        raise FileExistsError(f"Existing final file differs. It was preserved. New validated candidate: {submission}")
else:
    with submission.open("rb") as source, final_submission.open("xb") as target:
        shutil.copyfileobj(source, target)
assert hashlib.sha256(final_submission.read_bytes()).hexdigest() == validation["sha256"]
print("Complete, structurally valid file:", final_submission)
print("Rows:", validation["prediction_rows"], "SHA-256:", validation["sha256"])
print(json.dumps(summary, indent=2))
print("These are timing/format diagnostics, not a leaderboard score. Failed/timed-out samples may score zero.")


## 6. Download and submit manually

Download the file below or find `result/submission.jsonl` in Kaggle's output file browser. Save a notebook version to retain the file and reports. Upload the JSONL to the organisers' submission interface yourself; this notebook does not make a submission.

Keep the notebook and diagnostic outputs private. No official score is available locally for public_test. Source commit, model revisions, settings, errors, and timings are saved under `public_runs/` for reproducibility.


In [ ]:
from IPython.display import FileLink, display
print("Submission:", final_submission)
display(FileLink(str(final_submission)))
print("Run report:", run_dir / "run_report.json")
print("Validation:", validation_path)
print("Runtime:", run_dir / "runtime_summary.json")
